# WAP 품질 검증 — Jupyter (staging audit)

WAP 의 **audit** 단계는 결국 GE + Spark 일관성 체크다. CLI(`headway_wap.py --stage audit`)는
Airflow 가 publish 를 막는 **자동 게이트**(실패 시 `exit 1`)이고, 이 노트북은 **같은 검증을
사람이 대화형으로 시연/탐색**하는 용도다. 둘 다 `labs/16-data-quality/data_quality_checks.py`
의 동일 규칙을 쓴다 → 결과가 일치한다.

**전제:** staging 이 있어야 함 → 먼저 `bash scripts/ops/wap.sh write` 로 `iceberg.staging.*` 생성.


In [1]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder.appName('wap-audit-nb')
    .config('spark.sql.catalog.iceberg','org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.iceberg.type','rest')
    .config('spark.sql.catalog.iceberg.uri','http://iceberg-rest:8181')
    .config('spark.sql.catalog.iceberg.io-impl','org.apache.iceberg.aws.s3.S3FileIO')
    .config('spark.sql.catalog.iceberg.s3.endpoint','http://minio:9000')
    .config('spark.sql.catalog.iceberg.s3.path-style-access','true')
    .config('spark.sql.catalog.iceberg.warehouse','s3://warehouse/')
    .config('spark.sql.extensions','org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 07:14:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/26 07:14:01 WARN SparkSession: Cannot use org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions to configure session extensions.
java.lang.ClassNotFoundException: org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions
	at java.base/java.net.URLClassLoader.findClass(Unknown Source)
	at java.base/java.lang.ClassLoader.loadClass(Unknown Source)
	at java.base/java.lang.ClassLoader.loadClass(Unknown Source)
	at java.base/java.lang.Class.forName0(Native Method)
	at java.base/java.lang.Class.forName(Unknown Source)
	at org.apache.spark.util.SparkClassUtils.classForName(SparkClassUtils.scala:41)
	at org.apache.spark.util.SparkClassUtils.classForName$(SparkClassUtils.scala:36)
	at org.apache.spark.

## ① GE + Spark 일관성 체크 (staging) — audit 와 동일


In [2]:
import sys, pandas as pd
sys.path.insert(0, '/workspace/labs/16-data-quality')
import great_expectations as gx
from data_quality_checks import TABLE_KEYS, run_ge_suite, summarize_result, run_custom_checks

context = gx.get_context(mode='ephemeral')
ok = True
for key in TABLE_KEYS:
    res = run_ge_suite(context, spark, key, ns='staging')
    ok = ok and bool(res.success)
    print(f'[{key}] GE success = {res.success}')
    display(pd.DataFrame(summarize_result(res)))


Py4JJavaError: An error occurred while calling o40.table.
: org.apache.spark.SparkException: Cannot find catalog plugin class for catalog 'iceberg': org.apache.iceberg.spark.SparkCatalog.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.catalogPluginClassNotFoundForCatalogError(QueryExecutionErrors.scala:1926)
	at org.apache.spark.sql.connector.catalog.Catalogs$.load(Catalogs.scala:70)
	at org.apache.spark.sql.connector.catalog.CatalogManager.$anonfun$catalog$1(CatalogManager.scala:54)
	at scala.collection.mutable.HashMap.getOrElseUpdate(HashMap.scala:86)
	at org.apache.spark.sql.connector.catalog.CatalogManager.catalog(CatalogManager.scala:54)
	at org.apache.spark.sql.connector.catalog.LookupCatalog$CatalogAndIdentifier$.unapply(LookupCatalog.scala:122)
	at org.apache.spark.sql.catalyst.analysis.Analyzer$ResolveRelations$.$anonfun$resolveRelation$1(Analyzer.scala:1297)
	at scala.Option.orElse(Option.scala:447)
	at org.apache.spark.sql.catalyst.analysis.Analyzer$ResolveRelations$.

In [ ]:
print('Spark 직접 일관성 체크 (staging)')
checks = run_custom_checks(spark, ns='staging')
ok = ok and all(c['violations']==0 for c in checks)
display(pd.DataFrame(checks))
print('audit 종합:', 'PASS → publish 가능' if ok else 'FAIL → publish 중단(gold 유지)')


## ② publish 게이트 — 검증이 오염을 막아내는가

CLI 에선 위 검증이 하나라도 실패하면 `sys.exit(1)` → publish 중단. 아래는 staging 의 한 컬럼을
일부러 오염시켜 **success=False** 가 되는 걸 보여준다(= 오염은 publish 를 통과 못 함).


In [ ]:
from pyspark.sql import functions as F
from data_quality_checks import STAGING, expectations_for

bad = (spark.table(STAGING['headway'])
       .withColumn('cv', F.lit(99.0))        # cv in [0,3] 위반
       .withColumn('direction', F.lit('X'))) # direction in {상행,하행} 위반
ds = context.data_sources.add_spark(name='sp_bad_stage')
asset = ds.add_dataframe_asset(name='bad_stage')
bd = asset.add_batch_definition_whole_dataframe(name='whole')
suite = context.suites.add(gx.ExpectationSuite(name='bad_stage_suite'))
for e in expectations_for('headway'): suite.add_expectation(e)
bad_res = bd.get_batch(batch_parameters={'dataframe': bad}).validate(suite)
print('오염 staging success =', bad_res.success,
      '→', '정상: publish 차단' if not bad_res.success else '⚠ 못 막음')
display(pd.DataFrame(summarize_result(bad_res)))
